**Task 1: Data preprocessing and cleanup**

In [0]:
import pandas as pd #calling in the data manipulation library for usage 
import numpy as np

patient_readmission_spark=spark.read.table('mthimunye.default.expanded_patient_readmission_data_')

patient_readmission=patient_readmission_spark.toPandas()

display(patient_readmission.head(5))

In [0]:
patient_readmission.info()

# the output shows that we have zero nulls in each column, with correct dtypes for each column
# there are 10 columns and 3000 rows

In [0]:
patient_readmission.describe()

In [0]:
patient_readmission.duplicated().sum()
 #the output reveal 0 duplicates count

In [0]:
patient_readmission['Gender'].unique()

In [0]:
patient_readmission['Readmission'].unique()

In [0]:
patient_readmission['Admission Type'].unique()


**Checking that numeric values fall within plausible ranges (example an age of 0 or a stay of 400 days would an explanation)**

In [0]:
patient_readmission['Age'].unique()

#output is corresponding with the data dictionary with age (18-89)

In [0]:
patient_readmission['Length of Stay'].unique()

#output is corresponding with the data dictionary with length of stay (1-29) days


In [0]:
patient_readmission['Number of Diagnoses'].unique()
#output is corresponding with the data dictionary with number of diagnoses (1-9) days

In [0]:
patient_readmission['Blood Pressure'].unique()

#output is corresponding with the data dictionary with blood pressure (60-179) mmHg


In [0]:
patient_readmission['Blood Sugar Levels'].unique()
#output is corresponding with the data dictionary with blood sugar levels(51-199)mg/dL

In [0]:
patient_readmission['Previous Admissions'].unique()
##output is corresponding with the data dictionary with previous admissions (0-4)

In [0]:
#To avoid the test set leak, all imputers, scalers, and encoders are wrapped into a scikit-learn ColumnTransformer and fit strictly on the training data (X_train).

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. Separate features and target
numeric_features = [
    "Age",
    "Length of Stay",
    "Number of Diagnoses",
    "Blood Pressure",
    "Blood Sugar Levels",
    "Previous Admissions",
]
categorical_features = ["Gender", "Admission Type"]

X = patient_readmission[numeric_features + categorical_features]
y = (patient_readmission["Readmission"] == "Yes").astype(int)

# 2. Perform Test Split (80% train, 20% test) BEFORE fitting any transformation
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

# 3. Construct preprocessing pipelines per feature type
numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),  # Safe rule for future data exports
        ("scaler", StandardScaler()),  # Standardize to zero mean, unit variance
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(drop="first", handle_unknown="ignore"),
        ),  # Leak-free one-hot encoding
    ]
)

# 4. Combine into a unified ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# 5. Fit transformer ONLY on training set and transform both sets
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"X_train preprocessed shape: {X_train_processed.shape}")
print(f"X_test preprocessed shape:  {X_test_processed.shape}")

The output of display(X_train_processed) reveals that the preprocessing pipeline has successfully transformed the training dataset into a clean, model-ready numeric array with dimensions of 2,400 rows and 8 columns (features)

The output of display(X_test_processed) reveals that the preprocessing pipeline has successfully transformed the testing dataset into a clean, model-ready numeric array with dimensions of 600 rows and 8 columns (features)

**Task 2: Exploratory Data Analysis**

In [0]:
!pip install seaborn

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_theme(style="whitegrid")
plt.figure(figsize=(6, 5))

# Calculate percentage proportions
readmit_counts = patient_readmission["Readmission"].value_counts(
    normalize=True
) * 100

# Create bar plot
ax = sns.barplot(
    x=readmit_counts.index, y=readmit_counts.values, palette="Blues_d"
)
plt.title(
    "30-Day Patient Readmission Proportion",
    fontsize=12,
    fontweight="bold",
)
plt.xlabel("Readmission within 30 Days", fontsize=10)
plt.ylabel("Percentage (%)", fontsize=10)
plt.ylim(0, 100)

# Annotate percentage values on top of bars
for p in ax.patches:
  ax.annotate(
      f"{p.get_height():.1f}%",
      (p.get_x() + p.get_width() / 2.0, p.get_height() + 2),
      ha="center",
      va="bottom",
      fontsize=10,
  )

plt.tight_layout()
plt.show()

In [0]:


plt.figure(figsize=(7, 5))

# Create box plot
sns.boxplot(
    x="Readmission",
    y="Length of Stay",
    data=patient_readmission,
    palette="Set2",
)
plt.title(
    "Length of Stay vs. 30-Day Readmission",
    fontsize=12,
    fontweight="bold",
)
plt.xlabel("Readmission within 30 Days", fontsize=10)
plt.ylabel("Length of Stay (Days)", fontsize=10)

plt.tight_layout()
plt.show()

 
Interpretation :
 both groups show similar patterns, with hospital stays ranging from approximately 1 to 29 days. This suggests that there is no clear difference in the length of stay between the two groups. Therefore, length of stay alone may not be a strong predictor of patient readmission, and other patient characteristics should also be considered when developing the machine learning model.

In [0]:

# Calculate readmission rate percentage by admission type
admission_rate = (
    patient_readmission.groupby("Admission Type")["Readmission"]
    .apply(lambda x: (x == "Yes").mean() * 100)
    .reset_index()
)

# Create bar plot
ax = sns.barplot(
    x="Admission Type", y="Readmission", data=admission_rate, palette="muted"
)
plt.title(
    "Readmission Rate by Admission Type",
    fontsize=12,
    fontweight="bold",
)
plt.xlabel("Admission Type", fontsize=10)
plt.ylabel("Readmission Rate (%)", fontsize=10)
plt.ylim(0, max(admission_rate["Readmission"]) * 1.2)

# Annotate percentage values on top of bars
for p in ax.patches:
  ax.annotate(
      f"{p.get_height():.1f}%",
      (p.get_x() + p.get_width() / 2.0, p.get_height() + 1),
      ha="center",
      va="bottom",
      fontsize=10,
  )

plt.tight_layout()
plt.show()

In [0]:


# Select numeric columns for correlation matrix
numeric_cols = [
    "Age",
    "Length of Stay",
    "Number of Diagnoses",
    "Blood Pressure",
    "Blood Sugar Levels",
    "Previous Admissions",
]
corr_matrix = patient_readmission[numeric_cols].corr()

# Create heatmap
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    cbar_kws={"label": "Correlation Coefficient"},
)
plt.title(
    "Inter-Predictor Correlation Matrix", fontsize=12, fontweight="bold"
)

plt.tight_layout()
plt.show()

In [0]:


# Calculate readmission rate percentage by gender
gender_rate = (
    patient_readmission.groupby("Gender")["Readmission"]
    .apply(lambda x: (x == "Yes").mean() * 100)
    .reset_index()
)

# Create bar plot
ax = sns.barplot(
    x="Gender", y="Readmission", data=gender_rate, palette="Set2"
)
plt.title(
    "Readmission Rate by Gender", fontsize=12, fontweight="bold"
)
plt.xlabel("Gender", fontsize=10)
plt.ylabel("Readmission Rate (%)", fontsize=10)
plt.ylim(0, max(gender_rate["Readmission"]) * 1.2)

# Annotate percentage values on top of bars
for p in ax.patches:
  ax.annotate(
      f"{p.get_height():.1f}%",
      (p.get_x() + p.get_width() / 2.0, p.get_height() + 1),
      ha="center",
      va="bottom",
      fontsize=10,
  )

plt.tight_layout()
plt.show()

**Task 3: Feature Section**

In [0]:

# Select features and target

X = patient_readmission.drop(columns=["Patient ID", "Readmission"])
y = patient_readmission["Readmission"]

print("Input features:")
print(X.columns)

print("\nTarget values:")
print(y.value_counts())

Interpretation: The input features contain patient information such as age, gender, length of stay and previous admissions. The target variable is readmission (Yes or No). Patient ID is excluded because it is only an identifier and does not describe a patient's health.

In [0]:

# Convert categorical variables into numerics. (ENCODING)

X = pd.get_dummies(X, columns=["Gender", "Admission Type"], drop_first=True, dtype=int)

# Convert target variable
y = y.map({"No": 0, "Yes": 1})

print(X.head())

Interpretation: Categorical information has been converted into numerical values using Binary Mapping. For the target variable, 0 represents patients who were not readmitted, while 1 represents patients who were readmitted.

In [0]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Interpretation: The dataset is divided into training and testing sets. The model learns from 80% of the data, while the remaining 20% is used to evaluate its performance on unseen patients. The stratify=y option keeps the readmission proportions similar in both sets.

In [0]:

from sklearn.preprocessing import StandardScaler

numeric_cols = [
    "Age", "Length of Stay", "Number of Diagnoses",
    "Blood Pressure", "Blood Sugar Levels",
    "Previous Admissions"
]

scaler = StandardScaler()

X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

print(X_train.head())

Interpretation: The numerical variables have been standardised so that differences in their measurement scales do not unfairly influence certain models, particularly Logistic Regression. The scaler was fitted only on the training data to avoid using information from the test data during training.

In [0]:

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(random_state=42, class_weight="balanced")
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(name, "trained successfully")

Interpretation: All three models have been trained using historical patient data.

-Logistic Regression estimates the probability of readmission
-Decision Tree makes predictions using a series of decisions
-Random Forest combines several decision trees to make predictions.

In [0]:
#Cross-validation tests the models on different portions of the training data to see how consistently they perform.

from sklearn.model_selection import cross_val_score

for name, model in models.items():
    scores = cross_val_score(
        model, X_train, y_train,
        cv=5, scoring="roc_auc"
    )

    print(name)
    print("Mean ROC-AUC:", round(scores.mean(), 3))
    print("Standard deviation:", round(scores.std(), 3))
    print()

Interpretation:


The results show that Random Forest achieved the highest mean ROC-AUC score of 0.512, followed by Logistic Regression and Decision Tree, both scoring 0.499. However, all three models performed close to 0.5, which indicates that they are not much better than random guessing at identifying patients who may be readmitted within 30 days.

Random Forest also had the lowest standard deviation of 0.017, suggesting slightly more consistent performance across the five cross-validation rounds.

Although Random Forest performed slightly better than the other models, its predictive performance remains weak. This suggests that the available patient information may not be sufficient to accurately predict readmission.


In [0]:
#A baseline model provides a simple reference to determine whether the machine learning models are actually useful.

from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy="prior")
baseline_scores = cross_val_score(
    baseline, X_train, y_train,
    cv=5, scoring="roc_auc"
)

print("Baseline ROC-AUC:", round(baseline_scores.mean(), 3))

In [0]:
# We can improve the models by testing a small number of settings using GridSearchCV.


from sklearn.model_selection import GridSearchCV

# Tune Decision Tree
tree_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42, class_weight="balanced"),
    {"max_depth": [3, 5, 10]},
    cv=5,
    scoring="roc_auc"
)
tree_grid.fit(X_train, y_train)

print("Best Decision Tree settings:", tree_grid.best_params_)
print("Best score:", round(tree_grid.best_score_, 3))

# Tune Random Forest
forest_grid = GridSearchCV(
    RandomForestClassifier(random_state=42, class_weight="balanced"),
    {"max_depth": [5, 10], "n_estimators": [50, 100]},
    cv=5,
    scoring="roc_auc"
)
forest_grid.fit(X_train, y_train)

print("Best Random Forest settings:", forest_grid.best_params_)
print("Best score:", round(forest_grid.best_score_, 3))


**Task 5: Model Evaluation**

In [0]:

# Compare candidate models

candidates = {
    "Logistic Regression": models["Logistic Regression"],
    "Decision Tree": models["Decision Tree"],
    "Random Forest": models["Random Forest"],
    "Tuned Decision Tree": tree_grid.best_estimator_,
    "Tuned Random Forest": forest_grid.best_estimator_
}

results = {}

for name, model in candidates.items():
    scores = cross_val_score(
        model, X_train, y_train,
        cv=5, scoring="roc_auc"
    )
    results[name] = scores.mean()

best_name = max(results, key=results.get)
best_model = candidates[best_name]

print("Selected model:", best_name)
print("Cross-validation ROC-AUC:", round(results[best_name], 3))

In [0]:

from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score
)

y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("Precision:", round(precision_score(y_test, y_pred, zero_division=0), 3))
print("Recall:", round(recall_score(y_test, y_pred), 3))
print("F1 Score:", round(f1_score(y_test, y_pred), 3))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 3))

The model achieved an accuracy of 55.7%, meaning it correctly predicted approximately 56% of patient outcomes. The precision of 29.1% indicates that only about 29% of patients predicted to be readmitted were actually readmitted. The recall of 37.6% shows that the model identified only 37.6% of patients who were readmitted, missing the remaining 62.4%.

The low F1 score of 0.328 indicates a poor balance between precision and recall. Furthermore, the ROC-AUC score of 0.512 suggests that the model performs only slightly better than random guessing.

Overall, the model has weak predictive performance and is not yet reliable enough to identify patients at risk of readmission.

In [0]:

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

# Train Random Forest
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

# Get feature importance scores
importance = rf.feature_importances_

# Plot feature importance
plt.figure(figsize=(10, 5))
plt.bar(X_train.columns, importance)

plt.xlabel("Features")
plt.ylabel("Importance Score")
plt.title("Random Forest Feature Importance")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

The Random Forest feature importance graph shows which patient characteristics contributed most to the model's predictions. Features with higher importance scores had a greater influence on the model, while features with lower scores contributed less. However, feature importance does not necessarily mean that a variable accurately predicts patient readmission